# Day 7: 周复习与实战 - 日志记录器

## 学习内容
综合本周所学，实现一个带配置的日志记录器

## 1. 日志级别定义

In [ ]:
class LogLevel:
    """日志级别常量"""
    DEBUG = 0
    INFO = 1
    WARNING = 2
    ERROR = 3
    CRITICAL = 4

    @staticmethod
    def get_level_name(level):
        """获取日志级别名称"""
        names = {
            0: "DEBUG",
            1: "INFO",
            2: "WARNING",
            3: "ERROR",
            4: "CRITICAL"
        }
        return names.get(level, "UNKNOWN")

    @staticmethod
    def get_level_from_name(name):
        """从名称获取日志级别"""
        names = {
            "DEBUG": 0,
            "INFO": 1,
            "WARNING": 2,
            "ERROR": 3,
            "CRITICAL": 4
        }
        return names.get(name.upper(), 1)  # 默认 INFO

# 测试
print(f"DEBUG 级别数值：{LogLevel.DEBUG}")
print(f"级别 2 的名称：{LogLevel.get_level_name(2)}")
print(f"ERROR 的级别数值：{LogLevel.get_level_from_name('ERROR')}")

## 2. 日志记录器类

In [ ]:
import json
import os
from datetime import datetime

class Logger:
    """自定义日志记录器"""

    def __init__(self, name="app", log_file=None, level=LogLevel.INFO, config_file=None):
        """
        初始化日志记录器
        
        Args:
            name: 日志记录器名称
            log_file: 日志文件路径（可选）
            level: 日志级别
            config_file: 配置文件路径（可选）
        """
        self.name = name
        self.level = level
        self.log_file = log_file
        self.logs = []  # 内存中存储日志

        # 如果提供了配置文件，从文件加载配置
        if config_file:
            self.load_config(config_file)

        # 如果指定了日志文件，确保目录存在
        if log_file:
            log_dir = os.path.dirname(log_file)
            if log_dir and not os.path.exists(log_dir):
                os.makedirs(log_dir)

    def load_config(self, config_file):
        """从配置文件加载设置"""
        try:
            if os.path.exists(config_file):
                with open(config_file, 'r', encoding='utf-8') as f:
                    config = json.load(f)

                self.name = config.get('name', self.name)
                self.log_file = config.get('log_file', self.log_file)
                level_name = config.get('level', 'INFO')
                self.level = LogLevel.get_level_from_name(level_name)

                print(f"✅ 已加载配置：{config_file}")
        except Exception as e:
            print(f"⚠️ 加载配置失败：{e}")

    def save_config(self, config_file):
        """保存配置到文件"""
        config = {
            "name": self.name,
            "log_file": self.log_file,
            "level": LogLevel.get_level_name(self.level)
        }

        try:
            with open(config_file, 'w', encoding='utf-8') as f:
                json.dump(config, f, ensure_ascii=False, indent=2)
            print(f"✅ 配置已保存：{config_file}")
        except Exception as e:
            print(f"❌ 保存配置失败：{e}")

    def _format_message(self, level, message):
        """格式化日志消息"""
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        level_name = LogLevel.get_level_name(level)
        return f"[{timestamp}] [{level_name}] [{self.name}] {message}"

    def _log(self, level, message):
        """内部日志方法"""
        # 检查日志级别
        if level < self.level:
            return

        # 格式化消息
        formatted = self._format_message(level, message)

        # 存储到内存
        self.logs.append({
            "timestamp": datetime.now().isoformat(),
            "level": LogLevel.get_level_name(level),
            "message": message
        })

        # 输出到控制台
        print(formatted)

        # 写入文件
        if self.log_file:
            try:
                with open(self.log_file, 'a', encoding='utf-8') as f:
                    f.write(formatted + "\n")
            except Exception as e:
                print(f"[ERROR] 写入日志文件失败：{e}")

    def debug(self, message):
        """DEBUG 级别日志"""
        self._log(LogLevel.DEBUG, message)

    def info(self, message):
        """INFO 级别日志"""
        self._log(LogLevel.INFO, message)

    def warning(self, message):
        """WARNING 级别日志"""
        self._log(LogLevel.WARNING, message)

    def error(self, message):
        """ERROR 级别日志"""
        self._log(LogLevel.ERROR, message)

    def critical(self, message):
        """CRITICAL 级别日志"""
        self._log(LogLevel.CRITICAL, message)

    def get_logs(self, level=None, limit=None):
        """
        获取日志记录
        
        Args:
            level: 过滤级别（可选）
            limit: 限制数量（可选）
        
        Returns:
            日志列表
        """
        logs = self.logs

        if level:
            logs = [log for log in logs if log["level"] == level]

        if limit:
            logs = logs[-limit:]

        return logs

    def clear_logs(self):
        """清空内存中的日志"""
        self.logs = []
        self.info("日志已清空")

    def export_logs(self, output_file):
        """导出日志到文件"""
        try:
            with open(output_file, 'w', encoding='utf-8') as f:
                json.dump(self.logs, f, ensure_ascii=False, indent=2)
            self.info(f"日志已导出：{output_file}")
        except Exception as e:
            self.error(f"导出日志失败：{e}")

## 3. 基础日志演示

In [ ]:
# 创建日志记录器
logger = Logger(name="MyApp", level=LogLevel.DEBUG)

print("记录不同级别的日志:")
logger.debug("这是一条 DEBUG 日志")
logger.info("这是一条 INFO 日志")
logger.warning("这是一条 WARNING 日志")
logger.error("这是一条 ERROR 日志")
logger.critical("这是一条 CRITICAL 日志")

print(f"\n内存中的日志数量：{len(logger.get_logs())}")

print(f"\nERROR 级别的日志:")
for log in logger.get_logs(level="ERROR"):
    print(f"  - {log['message']}")

## 4. 不同日志级别测试

In [ ]:
# 创建只记录 WARNING 及以上级别的日志器
warning_logger = Logger(name="WarningOnly", level=LogLevel.WARNING)

print("使用 WARNING 级别的日志器:")
print("以下只有 WARNING 和 ERROR 会输出:\n")

warning_logger.debug("这条 DEBUG 不会显示")
warning_logger.info("这条 INFO 不会显示")
warning_logger.warning("⚠️ 这条 WARNING 会显示")
warning_logger.error("❌ 这条 ERROR 会显示")

## 5. 业务场景：用户管理系统

In [ ]:
# 创建日志记录器
logger = Logger(name="UserSystem", level=LogLevel.DEBUG)

class UserService:
    """用户服务类（带日志）"""
    
    def __init__(self, logger):
        self.logger = logger
        self.users = {}
        self.logger.info("用户服务初始化完成")

    def register(self, username, email):
        """用户注册"""
        self.logger.debug(f"尝试注册用户：{username}")

        if not username or len(username) < 3:
            self.logger.warning(f"用户名太短：{username}")
            return False, "用户名至少 3 个字符"

        if '@' not in email:
            self.logger.error(f"邮箱格式错误：{email}")
            return False, "邮箱格式不正确"

        if username in self.users:
            self.logger.warning(f"用户已存在：{username}")
            return False, "用户已存在"

        self.users[username] = {"email": email, "status": "active"}
        self.logger.info(f"用户注册成功：{username}")
        return True, "注册成功"

    def login(self, username, password):
        """用户登录"""
        self.logger.debug(f"尝试登录：{username}")

        if username not in self.users:
            self.logger.warning(f"登录失败 - 用户不存在：{username}")
            return False, "用户不存在"

        if len(password) < 6:
            self.logger.error(f"登录失败 - 密码太短：{username}")
            return False, "密码错误"

        self.logger.info(f"用户登录成功：{username}")
        return True, "登录成功"

    def delete_user(self, username):
        """删除用户"""
        self.logger.info(f"尝试删除用户：{username}")

        if username not in self.users:
            self.logger.warning(f"删除失败 - 用户不存在：{username}")
            return False

        del self.users[username]
        self.logger.info(f"用户已删除：{username}")
        return True

In [ ]:
# 运行业务场景
service = UserService(logger)

print("=" * 50)
print("1. 用户注册测试")
print("=" * 50)
service.register("ab", "test@example.com")  # 用户名太短
service.register("alice", "invalid-email")   # 邮箱格式错误
service.register("alice", "alice@example.com")  # 成功
service.register("alice", "alice2@example.com")  # 重复注册
service.register("bob", "bob@example.com")  # 成功

In [ ]:
print("\n" + "=" * 50)
print("2. 用户登录测试")
print("=" * 50)
service.login("alice", "123")    # 密码太短
service.login("unknown", "123456")  # 用户不存在
service.login("bob", "password123")  # 成功

In [ ]:
print("\n" + "=" * 50)
print("3. 用户删除测试")
print("=" * 50)
service.delete_user("nonexistent")
service.delete_user("alice")

print(f"\n当前用户列表：{list(service.users.keys())}")
print(f"总日志数：{len(logger.get_logs())}")

In [ ]:
# 查看日志历史
print("\n所有日志记录:")
for log in logger.get_logs():
    print(f"  [{log['level']}] {log['message']}")

In [ ]:
# 导出日志
logger.export_logs("logs_export.json")
print("日志已导出到 logs_export.json")

In [ ]:
# 清理导出文件
if os.path.exists("logs_export.json"):
    os.remove("logs_export.json")
    print("已清理测试文件")

## 6. 第一周学习总结

### 本周学习内容回顾

| 日期 | 学习内容 | 关键知识点 |
|------|----------|------------|
| Day 1 | Python 环境搭建 | 安装 Python、配置环境、hello.py |
| Day 2 | 基础语法 | 变量、字符串、列表、元组、字典 |
| Day 3 | 条件语句与循环 | if/elif/else、for/while、猜数字游戏 |
| Day 4 | 函数与模块 | 函数定义、参数、返回值、import、装饰器 |
| Day 5 | 异常处理 | try/except/finally、自定义异常 |
| Day 6 | 文件与 IO | 文件读写、JSON、配置文件读取器 |
| Day 7 | 综合实战 | 日志记录器完整实现 |

### 核心技能
- ✅ 能用 Python 写简单的脚本
- ✅ 理解变量、循环、条件等基础概念
- ✅ 能定义和使用函数
- ✅ 能处理异常和错误
- ✅ 能读写文件和 JSON 数据

### 下一步
准备进入第 2 周：Python 进阶与异步编程

---
✅ 第一周学习完成！🎉